# NLP – Tutorial 1 Lab: German Text Corpus
**Group 2 – German** · model `de_core_news_md`
**Members:** Blevis Allushi · Darsi Ramaj · Geri Belba

Run in **JupyterLab** (kernel with spaCy installed; see `README.md`). Order: **Kernel → Restart Kernel and Run All Cells**. Needs internet for the Wikipedia API and the one-time spaCy model download. If the setup cell had to install spaCy, restart the kernel once, then run all again.

| Part | Owner | Sections |
|---|---|---|
| Setup, pipeline demo, displaCy, Corpus collection, UTF-8 check | **Blevis** | 0, 1, 5 |
| Sources, licences, discussion, quality problems | **Geri** | 2, 4, 7 |
| Corpus statistics, quality problems, table row | **Darsi** | 3, 4, 6 |

## 0 · Setup and version printout  *(Blevis)*

In [17]:
# Installs into the *kernel's* environment (skip these two lines if you already followed README.md)
%pip install -q -U spacy requests pandas
import sys
!{sys.executable} -m spacy download de_core_news_md

Note: you may need to restart the kernel to use updated packages.
  Using cached de_core_news_md-3.8.0-py3-none-any.whl (44.4 MB)
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_md')


In [18]:
import re, csv, spacy, pandas as pd
from pathlib import Path
from collections import Counter
from spacy import displacy
from IPython.display import display, HTML, Markdown

print('spaCy version:', spacy.__version__)
nlp = spacy.load('de_core_news_md')
print('Model:', nlp.meta['name'], nlp.meta['version'], '| pipeline:', nlp.pipe_names)

# One sentence through the full pipeline
doc = nlp('Johann Wolfgang von Goethe wurde 1749 in Frankfurt am Main geboren.')
display(pd.DataFrame([(t.i, t.text, t.lemma_, t.pos_, t.tag_, t.dep_, t.head.text, t.ent_type_) for t in doc],
                     columns=['i','token','lemma','pos','tag','dep','head','ent']))
print('Entities:', [(e.text, e.label_) for e in doc.ents])

spaCy version: 3.8.18
Model: core_news_md 3.8.0 | pipeline: ['tok2vec', 'tagger', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


,i,token,lemma,pos,tag,dep,head,ent
0,0,Johann,Johann,PROPN,NE,pnc,Goethe,PER
1,1,Wolfgang,Wolfgang,PROPN,NE,pnc,Goethe,PER
2,2,von,von,PROPN,NE,pnc,Goethe,PER
3,3,Goethe,Goethe,PROPN,NE,sb,wurde,PER
4,4,wurde,werden,AUX,VAFIN,ROOT,wurde,
5,5,1749,1749,NUM,CARD,mo,geboren,
6,6,in,in,ADP,APPR,mo,geboren,
7,7,Frankfurt,Frankfurt,PROPN,NE,nk,in,LOC
8,8,am,an,ADP,APPRART,mnr,Frankfurt,LOC
9,9,Main,Main,PROPN,NE,nk,am,LOC


Entities: [('Johann Wolfgang von Goethe', 'PER'), ('Frankfurt am Main', 'LOC')]


In [19]:
displacy.render(doc, style='dep', jupyter=True)
displacy.render(doc, style='ent', jupyter=True)

## 1 · Collecting the texts  *(Blevis)*
Text type: **introductions of German Wikipedia articles** (CC BY-SA 4.0, reuse allowed with attribution). Fetched through the official MediaWiki API (no scraping). Each intro is cut at a sentence boundary to at most 300 words; articles with fewer than 100 words are skipped. Metadata (title, URL, retrieval date, last-modified date, licence) is written to `sources.csv`.

In [20]:
import requests, re, csv, datetime, time
from pathlib import Path

CORPUS = Path('corpus'); CORPUS.mkdir(exist_ok=True)
API = 'https://de.wikipedia.org/w/api.php'
HEADERS = {'User-Agent': 'UMT-NLP-Lab/1.0 (university course corpus; Wikipedia API, low volume)'}
LICENCE = 'CC BY-SA 4.0'
MAX_TEXTS, MIN_W, MAX_W = 30, 100, 300

# Mixed topics on purpose (cities, nature, people, companies) -> umlauts, ß, long compounds, many entities
TITLES = ['Berlin','München','Hamburg','Köln','Frankfurt am Main','Stuttgart','Düsseldorf','Leipzig','Dresden',
          'Nürnberg','Bremen','Hannover','Heidelberg','Rhein','Donau','Elbe','Bodensee','Schwarzwald','Zugspitze',
          'Harz','Johann Wolfgang von Goethe','Albert Einstein','Ludwig van Beethoven','Johann Sebastian Bach',
          'Immanuel Kant','Volkswagen','Siemens','Bundesrepublik Deutschland','Oktoberfest','Brandenburger Tor',
          'Rügen','Mosel (Fluss)','Lüneburger Heide','Weimar']

def fetch_intro(title):
    r = requests.get(API, headers=HEADERS, timeout=30, params=dict(
        action='query', format='json', prop='extracts|info', exintro=1, explaintext=1,
        redirects=1, inprop='url', titles=title))
    r.raise_for_status()
    return next(iter(r.json()['query']['pages'].values()))

def cut_to_words(text, max_w=MAX_W):
    text = re.sub(r'\s*\n+\s*', ' ', text).strip()
    out, n = [], 0
    for s in re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ])', text):
        w = len(s.split())
        if n + w > max_w: break
        out.append(s); n += w
    return ' '.join(out), n

rows, skipped = [], []
today = datetime.date.today().isoformat()
for title in TITLES:
    if len(rows) >= MAX_TEXTS: break
    try:
        p = fetch_intro(title)
        text, n = cut_to_words(p.get('extract', ''))
    except Exception as e:
        skipped.append((title, repr(e))); continue
    if n < MIN_W:
        skipped.append((title, f'only {n} words')); continue
    fname = f'text{len(rows)+1:02d}.txt'
    (CORPUS / fname).write_text(text, encoding='utf-8')
    rows.append(dict(file=fname, title=p['title'], url=p['fullurl'], retrieved=today,
                     last_modified=p.get('touched','')[:10], licence=LICENCE, words=n))
    time.sleep(0.3)

with open('sources.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print(f'Saved {len(rows)} texts. Skipped: {skipped}')
pd.DataFrame(rows)

Saved 18 texts. Skipped: [('Bremen', "HTTPError('429 Client Error: Too Many Requests for url: https://de.wikipedia.org/w/api.php?action=query&format=json&prop=extracts%7Cinfo&exintro=1&explaintext=1&redirects=1&inprop=url&titles=Bremen')"), ('Hannover', "HTTPError('429 Client Error: Too Many Requests for url: https://de.wikipedia.org/w/api.php?action=query&format=json&prop=extracts%7Cinfo&exintro=1&explaintext=1&redirects=1&inprop=url&titles=Hannover')"), ('Heidelberg', "HTTPError('429 Client Error: Too Many Requests for url: https://de.wikipedia.org/w/api.php?action=query&format=json&prop=extracts%7Cinfo&exintro=1&explaintext=1&redirects=1&inprop=url&titles=Heidelberg')"), ('Rhein', "HTTPError('429 Client Error: Too Many Requests for url: https://de.wikipedia.org/w/api.php?action=query&format=json&prop=extracts%7Cinfo&exintro=1&explaintext=1&redirects=1&inprop=url&titles=Rhein')"), ('Donau', "HTTPError('429 Client Error: Too Many Requests for url: https://de.wikipedia.org/w/api.php?ac

,file,title,url,retrieved,last_modified,licence,words
0,text01.txt,Berlin,https://de.wikipedia.org/wiki/Berlin,2026-11-08,2026-11-07,CC BY-SA 4.0,294
1,text02.txt,München,https://de.wikipedia.org/wiki/M%C3%BCnchen,2026-11-08,2026-11-06,CC BY-SA 4.0,171
2,text03.txt,Hamburg,https://de.wikipedia.org/wiki/Hamburg,2026-11-08,2026-11-06,CC BY-SA 4.0,211
3,text04.txt,Köln,https://de.wikipedia.org/wiki/K%C3%B6ln,2026-11-08,2026-11-06,CC BY-SA 4.0,293
4,text05.txt,Frankfurt am Main,https://de.wikipedia.org/wiki/Frankfurt_am_Main,2026-11-08,2026-11-06,CC BY-SA 4.0,284
5,text06.txt,Stuttgart,https://de.wikipedia.org/wiki/Stuttgart,2026-11-08,2026-11-06,CC BY-SA 4.0,212
6,text07.txt,Düsseldorf,https://de.wikipedia.org/wiki/D%C3%BCsseldorf,2026-11-08,2026-11-06,CC BY-SA 4.0,163
7,text08.txt,Leipzig,https://de.wikipedia.org/wiki/Leipzig,2026-11-08,2026-11-06,CC BY-SA 4.0,276
8,text09.txt,Dresden,https://de.wikipedia.org/wiki/Dresden,2026-11-08,2026-11-06,CC BY-SA 4.0,284
9,text10.txt,Nürnberg,https://de.wikipedia.org/wiki/N%C3%BCrnberg,2026-11-08,2026-11-06,CC BY-SA 4.0,295


In [21]:
# UTF-8 check: re-open files, look for mojibake and confirm special characters survive
CORPUS = Path('corpus')
MOJIBAKE = re.compile(r'Ã.|â€|\ufffd|Â')
bad = []
from collections import Counter
chars = Counter()
for f in sorted(CORPUS.glob('*.txt')):
    t = f.read_text(encoding='utf-8')
    if MOJIBAKE.search(t): bad.append(f.name)
    chars.update(c for c in t if c in 'äöüÄÖÜß')
print('Files with mojibake:', bad or 'none')
print('Special characters found:', dict(chars))
sample = next(s for f in sorted(CORPUS.glob('*.txt')) for s in f.read_text(encoding='utf-8').split('. ') if any(c in s for c in 'äöüß'))
print('Sample:', sample)

Files with mojibake: none
Special characters found: {'ß': 69, 'ö': 120, 'ä': 154, 'ü': 155, 'Ü': 6, 'Ö': 3, 'Ä': 1}
Sample: Die Großstadt ist mit rund 3,7 Millionen Einwohnern die bevölkerungsreichste und mit 891 Quadratkilometern die flächengrößte Gemeinde Deutschlands sowie die bevölkerungsreichste Stadt der Europäischen Union


## 2 · Sources and licences  *(Geri)*

In [22]:
src = pd.read_csv('sources.csv')
lines = [f'- **{r.title}** – <{r.url}> – retrieved {r.retrieved} (last modified {r.last_modified}) – {r.licence}' for r in src.itertuples()]
display(Markdown('All texts are introductions of German Wikipedia articles, licensed **CC BY-SA 4.0** (authors: see each article history).\n\n' + '\n'.join(lines)))

All texts are introductions of German Wikipedia articles, licensed **CC BY-SA 4.0** (authors: see each article history).

- **Berlin** – <https://de.wikipedia.org/wiki/Berlin> – retrieved 2026-11-08 (last modified 2026-11-07) – CC BY-SA 4.0
- **München** – <https://de.wikipedia.org/wiki/M%C3%BCnchen> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Hamburg** – <https://de.wikipedia.org/wiki/Hamburg> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Köln** – <https://de.wikipedia.org/wiki/K%C3%B6ln> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Frankfurt am Main** – <https://de.wikipedia.org/wiki/Frankfurt_am_Main> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Stuttgart** – <https://de.wikipedia.org/wiki/Stuttgart> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Düsseldorf** – <https://de.wikipedia.org/wiki/D%C3%BCsseldorf> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Leipzig** – <https://de.wikipedia.org/wiki/Leipzig> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Dresden** – <https://de.wikipedia.org/wiki/Dresden> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Nürnberg** – <https://de.wikipedia.org/wiki/N%C3%BCrnberg> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Johann Sebastian Bach** – <https://de.wikipedia.org/wiki/Johann_Sebastian_Bach> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Immanuel Kant** – <https://de.wikipedia.org/wiki/Immanuel_Kant> – retrieved 2026-11-08 (last modified 2026-11-05) – CC BY-SA 4.0
- **Siemens** – <https://de.wikipedia.org/wiki/Siemens> – retrieved 2026-11-08 (last modified 2026-11-05) – CC BY-SA 4.0
- **Deutschland** – <https://de.wikipedia.org/wiki/Deutschland> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Oktoberfest** – <https://de.wikipedia.org/wiki/Oktoberfest> – retrieved 2026-11-08 (last modified 2026-11-05) – CC BY-SA 4.0
- **Brandenburger Tor** – <https://de.wikipedia.org/wiki/Brandenburger_Tor> – retrieved 2026-11-08 (last modified 2026-11-06) – CC BY-SA 4.0
- **Rügen** – <https://de.wikipedia.org/wiki/R%C3%BCgen> – retrieved 2026-11-08 (last modified 2026-11-05) – CC BY-SA 4.0
- **Lüneburger Heide** – <https://de.wikipedia.org/wiki/L%C3%BCneburger_Heide> – retrieved 2026-11-08 (last modified 2026-11-05) – CC BY-SA 4.0

## 3 · Corpus statistics  *(Darsi)*

In [23]:
from collections import Counter
files = sorted(Path('corpus').glob('*.txt'))
texts = [f.read_text(encoding='utf-8') for f in files]
tokens = []
for d in nlp.pipe(texts):
    tokens += [t.text.lower() for t in d if t.is_alpha]

print('Texts:', len(texts))
print('Tokens:', len(tokens))
print('Unique words:', len(set(tokens)))
top10 = Counter(tokens).most_common(10)
print(top10)
sw = sum(c for w, c in Counter(tokens).items() if w in nlp.Defaults.stop_words)
print(f'Stop-word share of all tokens: {sw/len(tokens):.1%}')

Texts: 18
Tokens: 3842
Unique words: 1459
[('und', 191), ('der', 172), ('die', 143), ('ist', 80), ('des', 74), ('in', 66), ('mit', 62), ('das', 54), ('den', 46), ('stadt', 42)]
Stop-word share of all tokens: 45.6%


The 10 most frequent words are mostly German function words. Nine out of the ten words are classified as stop words by spaCy, including "und", "die", "der" and "ist". The only word that is not a stop word is "stadt", which provides more information about the topics covered in the corpus. The stop-word share of 45.6% confirms that common function words make up a large part of the corpus.

## 4 · Quality problems  *(Geri & Darsi)*

In [30]:
checks = {
  'HTML tags':            re.compile(r'<[^>]+>'),
  'HTML entities':        re.compile(r'&(?:[a-z]+|#\d+);'),
  'wiki markup (==, [[)': re.compile(r'==|\[\[|\]\]|\{\{'),
  'empty brackets ( ; )': re.compile(r'\(\s*[;,]?\s*\)'),
  'non-Latin script':     re.compile(r'[\u0370-\u03FF\u0400-\u04FF\u0590-\u06FF\u0900-\u0DFF\u3040-\u30FF\u4E00-\u9FFF]'),
  'IPA / pronunciation':  re.compile(r'[ɐ-˿]'),
  'boilerplate words':    re.compile(r'Cookie|Impressum|Navigation|Menü|Weiterleitung|Siehe auch|Einzelnachweise', re.I),
  'double spaces':        re.compile(r'  +'),
}
rep = {k: [] for k in checks}
for f, t in zip(files, texts):
    for k, rx in checks.items():
        m = rx.findall(t)
        if m: rep[k].append((f.name, len(m), m[0]))
for k, v in rep.items():
    print(f'{k:24s} files affected: {len(v)}', v[:3])

# repeated lines / repeated sentences across the corpus
sent_counts = Counter(s.strip() for t in texts for s in re.split(r'(?<=[.!?])\s+', t) if len(s.split()) > 4)
print('Repeated sentences:', [s for s, c in sent_counts.items() if c > 1][:5] or 'none')

# sentences that look English (very rough: >=2 common English function words)
EN = {'the','and','of','is','was','with','for','from','by','at'}
foreign = [(f.name, s) for f, t in zip(files, texts) for s in re.split(r'(?<=[.!?])\s+', t)
           if len(EN & set(re.findall(r'[a-z]+', s.lower()))) >= 2]
print('Possibly non-German sentences:', foreign[:5] or 'none')

HTML tags                files affected: 0 []
HTML entities            files affected: 0 []
wiki markup (==, [[)     files affected: 0 []
empty brackets ( ; )     files affected: 1 [('text05.txt', 1, '()')]
non-Latin script         files affected: 1 [('text03.txt', 1, 'χ')]
IPA / pronunciation      files affected: 5 [('text01.txt', 1, 'ɛ'), ('text02.txt', 9, 'ˈ'), ('text03.txt', 9, 'ˈ')]
boilerplate words        files affected: 0 []
double spaces            files affected: 2 [('text02.txt', 2, '  '), ('text08.txt', 2, '  ')]
Repeated sentences: none
Possibly non-German sentences: none


We checked the corpus automatically for HTML tags and entities, wiki markup, empty brackets, non-Latin script, IPA pronunciation, boilerplate words, double spaces, repeated sentences and English sentences.

- **No HTML or wiki markup** was found , because the API returns clean plain text.
- **Empty or leftover brackets:** Only 1 file has empty brackets  where Wikipedia templates (pronunciation, native names) were removed.
- **Pronunciation and name variants:** 5 files contain IPA pronunciation in brackets, for example in the Düsseldorf introduction ("[ˈdʏsl̩ˌdɔʁf]"), plus dialect names. These are not normal German text.

- **Non-Latin script:** Only 1 file has a non-latin script.
- **Double spaces** isseen on 2 of the files.
- **Repeated or English sentences:** No repeated sentences or english ones are in either of the files.

Overall the texts are clean, and the main problems are bracket artefacts and IPA in the first sentences of the articles.

The corpus contains no HTML tags, HTML entities or leftover Wikipedia markup. Minor problems were found, including empty brackets in text05.txt, a Greek character in text03.txt, IPA/pronunciation artefacts in five files and double spaces in two files. No repeated sentences or potentially non-German sentences were detected.

## 5 · Three sentences through the pipeline  *(Blevis)*

In [25]:
# Candidates picked automatically from the corpus; override SENTENCES with your own picks if you prefer.
pool = [s for d in nlp.pipe(texts) for s in d.sents if 8 <= len(s) <= 30]
A = max(pool, key=lambda s: len(s.ents))                                                    # most entities
B = max(pool, key=lambda s: max(len(t.text) for t in s if t.is_alpha))                      # longest word (compound)
CONTR = {'im','zum','zur','vom','beim','am','ins','ans','aufs','vom'}
C = next((s for s in pool if any(t.lower_ in CONTR for t in s) and s.text != A.text and s.text != B.text), pool[0])
SENTENCES = [A.text.strip(), B.text.strip(), C.text.strip()]
for i, s in enumerate(SENTENCES, 1): print(i, s, '\n')

1 Düsseldorf ([ˈdʏsl̩ˌdɔʁf] , Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirks Düsseldorf. 

2 Seine Werke beeinflussten spätere Komponistengenerationen und inspirieren bis heute musikschaffende Künstler zu zahllosen Bearbeitungen. 

3 Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder. 



In [26]:
Path('figures').mkdir(exist_ok=True)
for i, s in enumerate(SENTENCES, 1):
    d = nlp(s)
    print(f'=== Sentence {i} ===\n{s}')
    display(pd.DataFrame([(t.text, t.lemma_, t.pos_, t.dep_, t.head.text, t.ent_type_) for t in d],
                         columns=['token','lemma','pos','dep','head','ent']))
    svg = displacy.render(d, style='dep', jupyter=False, options={'distance': 90})
    Path(f'figures/sentence{i}_dep.svg').write_text(svg, encoding='utf-8')
    display(HTML(svg))
    displacy.render(d, style='ent', jupyter=True)

=== Sentence 1 ===
Düsseldorf ([ˈdʏsl̩ˌdɔʁf] , Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirks Düsseldorf.


,token,lemma,pos,dep,head,ent
0,Düsseldorf,Düsseldorf,PROPN,sb,ist,LOC
1,(,--,PUNCT,punct,[,
2,[,[,PROPN,par,Düsseldorf,
3,ˈdʏsl̩ˌdɔʁf,ˈdʏsl̩ˌdɔʁf,PROPN,punct,[,
4,],],PUNCT,punct,[,
5,",",--,PUNCT,punct,Düsseldorf,
6,Düsseldorfer,Düsseldorfer,ADJ,nk,Platt,MISC
7,Platt,Platt,NOUN,cj,Düsseldorf,LOC
8,Düsseldörp,Düsseldörp,PROPN,nk,Platt,LOC
9,),--,PUNCT,punct,Düsseldorf,


=== Sentence 2 ===
Seine Werke beeinflussten spätere Komponistengenerationen und inspirieren bis heute musikschaffende Künstler zu zahllosen Bearbeitungen.


,token,lemma,pos,dep,head,ent
0,Seine,sein,DET,nk,Werke,
1,Werke,Werk,NOUN,sb,beeinflussten,
2,beeinflussten,beeinflusst,ADJ,ROOT,beeinflussten,
3,spätere,spät,ADJ,nk,Komponistengenerationen,
4,Komponistengenerationen,Komponistengeneratione,NOUN,oa,beeinflussten,LOC
5,und,und,CCONJ,cd,beeinflussten,
6,inspirieren,inspirieren,VERB,cj,und,
7,bis,bis,ADP,mo,musikschaffende,
8,heute,heute,ADV,nk,bis,
9,musikschaffende,musikschaffend,ADJ,nk,Künstler,


=== Sentence 3 ===
Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.


,token,lemma,pos,dep,head,ent
0,Neben,neben,ADP,mnr,Fließgewässer,
1,den,der,DET,nk,Flüssen,
2,Flüssen,Fluß,NOUN,nk,Neben,
3,Spree,Spree,PROPN,nk,Flüssen,LOC
4,",",--,PUNCT,punct,Spree,
5,Havel,Havel,PROPN,cj,Spree,LOC
6,und,und,CCONJ,cd,Havel,
7,Dahme,Dahme,PROPN,cj,und,LOC
8,befinden,befinden,VERB,ROOT,befinden,
9,sich,sich,PRON,oa,befinden,


### Sentence 1:
Düsseldorf ([ˈdʏsl̩ˌdɔʁf], Düsseldorfer Platt Düsseldörp) ist die Landeshauptstadt Nordrhein-Westfalens und der Behördensitz des Regierungsbezirks Düsseldorf.

Correct: the core structure is right. Düsseldorf is the subject (sb) of ist, Nordrhein-Westfalens is a genitive attribute (ag) of Landeshauptstadt, and und correctly links the two nouns as a coordination (cd/cj).

Incorrect: Behördensitz is tagged PROPN (proper noun) but is a common noun, so it should be NOUN. German capitalizes all nouns, which probably misleads the tagger. The IPA pronunciation is also treated as a normal token (tagged PROPN, labelled punct), although it is not German text.

### Sentence 2:
Im Zuge der Montagsdemonstrationen 1989, die einen entscheidenden Impuls für den Zusammenbruch der DDR gaben, wurde Leipzig als Heldenstadt bezeichnet.

Correct: the long-distance structure is handled well. The relative pronoun die is the subject of gaben, the relative clause (rc) attaches to Montagsdemonstrationen, and the passive wurde … bezeichnet is linked correctly (oc).

Incorrect: I can’t find a clear error in the POS tags or dependencies. The only debatable spot is als Heldenstadt labelled mo (modifier). That is arguably a complement of bezeichnet, but I’m not sure enough to call it wrong. Send me the lemma/entity table for this sentence and I’ll check there. You can also swap in a different sentence, which is legitimate as long as you only report errors the pipeline actually made.

### Sentence 3:
Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.

Correct: the coordinations (Spree, Havel und Dahme, Fließgewässer sowie Seen und Wälder) are parsed correctly. The postposed subject Fließgewässer is attached to befinden, and sich is attached as its reflexive object.

Incorrect: the fronted phrase Neben den Flüssen … is attached to the noun Fließgewässer (mnr, postnominal modifier). It belongs to the verb befinden, because in German the phrase before the finite verb is part of the verb’s clause. A noun that comes after the verb cannot own it.

## 6 · German-specific behaviour and our table row  *(Darsi)*

In [27]:
tests = ['Kraftfahrzeug-Haftpflichtversicherung', 'Donaudampfschifffahrtsgesellschaft', 'zum Bahnhof', 'im Haus', 'vom Rhein', 'Straße', 'Müller-Lüdenscheidt']
for s in tests:
    d = nlp(s)
    print(f'{s!r:42} ->', [(t.text, t.lemma_, t.pos_) for t in d])

'Kraftfahrzeug-Haftpflichtversicherung'    -> [('Kraftfahrzeug-Haftpflichtversicherung', 'Kraftfahrzeug-Haftpflichtversicherung', 'NOUN')]
'Donaudampfschifffahrtsgesellschaft'       -> [('Donaudampfschifffahrtsgesellschaft', 'Donaudampfschifffahrtsgesellschaft', 'NOUN')]
'zum Bahnhof'                              -> [('zum', 'zu', 'ADP'), ('Bahnhof', 'Bahnhof', 'NOUN')]
'im Haus'                                  -> [('im', 'in', 'ADP'), ('Haus', 'Haus', 'NOUN')]
'vom Rhein'                                -> [('vom', 'von', 'ADP'), ('Rhein', 'Rhein', 'PROPN')]
'Straße'                                   -> [('Straße', 'Straße', 'NOUN')]
'Müller-Lüdenscheidt'                      -> [('Müller-Lüdenscheidt', 'Müller-Lüdenscheidt', 'PROPN')]


In [28]:

row = pd.DataFrame([dict(Language='German', Source='de.wikipedia.org (article introductions)', Licence='CC BY-SA 4.0',
                         Texts=len(texts), Tokens=len(tokens), **{'Unique words': len(set(tokens))},
                         **{'Problems found': 'Empty brackets in 1 file, Greek character in 1 file, IPA/pronunciation artefacts in 5 files and double spaces in 2 files.'})])
display(row)


,Language,Source,Licence,Texts,Tokens,Unique words,Problems found
0,German,de.wikipedia.org (article introductions),CC BY-SA 4.0,18,3842,1459,FILL IN from section 4


**1. Problems & causes:**

The corpus contains minor formatting and text-quality problems. These include empty brackets, a Greek character, IPA/pronunciation artefacts and double spaces. No HTML tags, leftover Wikipedia markup, repeated sentences or potentially non-German sentences were detected.

**2. Long words / accents / contractions:**

German compound words remain single tokens, including long and hyphenated compounds. The tokenizer does not automatically split them into their individual components. German umlauts (ä, ö, ü) and ß are preserved correctly. Contractions such as "zum", "im" and "vom" remain single tokens, but the lemmatizer maps them to "zu", "in" and "von", respectively.

**3. What to change before real use:**

Before using the corpus for further NLP analysis, empty brackets, IPA artefacts and unnecessary whitespace should be cleaned. Near-duplicate texts should also be checked, and topic diversity and metadata should be maintained.

## 7. Discussion (Geri)

**Is the corpus suitable?** Wikipedia introductions are a good starting point: they are free to reuse, well structured, correctly encoded and contain many entities, umlauts, ß and long compounds. But they are not representative of everyday German. The register is formal and encyclopedic, sentences are long and nominal, and our topic list (cities, rivers, historical people, companies) is chosen by us, so words like *Stadt*, *Deutschland* and *Fluss* are over-represented. The 18 texts of about 300 words are also very small for statistics.

**Data problems.** The main issues are IPA pronunciation, dialect and name variants in brackets, and empty brackets left by removed templates (see section 4). They matter because spaCy treats them as normal tokens, as we saw in the Düsseldorf sentence, where the IPA string was tagged as a proper noun. Before real use we would remove bracketed pronunciation, delete empty brackets, and check for duplicate or near-identical introductions.

**spaCy errors (section 5).** The model handled the main sentence structure well, including subjects, genitive attributes, coordination and relative clauses. Typical errors were capitalised common nouns tagged as PROPN instead of NOUN (German capitalises all nouns, which can mislead the tagger), and a fronted phrase attached to a noun instead of the verb. 

**German-specific behaviour.** Compounds such as *Kraftfahrzeug-Haftpflichtversicherung* stay a single token, so there is no decompounding, which makes the vocabulary larger and rarer words more common. Contractions like *zum*, *im* and *vom* are not split into *zu dem*, *in dem*, *von dem*, but the lemmatizer maps them to *zu*, *in* and *von*. Umlauts and ß are handled correctly, and our UTF-8 check found no unreadable text.

**What we would change before real use:** use more texts and more text types (news, forum text, fiction) for balance, remove bracket and IPA artefacts, remove duplicates, keep the per-text metadata and licence information, and manually check a sample of the annotation to measure the model's error rate.